In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

In [2]:
# Liquid properties
liquids = ['Water', 'Ethanol', 'Propylene_Glycol', 'Mercury', 'Toluene', 'Gasoline', 'Kerosene', 'Benzene', 'Acetone', 'Methanol']
density_ref = {
    273: {
        'Water': 999.8, 'Ethanol': 806.0, 'Propylene_Glycol': 1056.0, 'Mercury': 13600.0, 'Toluene': 884.0,
        'Gasoline': 740.0, 'Kerosene': 820.0, 'Benzene': 899.0, 'Acetone': 804.0, 'Methanol': 810.0
    },
    293: {
        'Water': 998.2, 'Ethanol': 789.0, 'Propylene_Glycol': 1040.0, 'Mercury': 13546.0, 'Toluene': 866.9,
        'Gasoline': 730.0, 'Kerosene': 810.0, 'Benzene': 879.0, 'Acetone': 791.0, 'Methanol': 792.0
    },
    313: {
        'Water': 992.2, 'Ethanol': 772.0, 'Propylene_Glycol': 1024.0, 'Mercury': 13492.0, 'Toluene': 849.8,
        'Gasoline': 720.0, 'Kerosene': 800.0, 'Benzene': 859.0, 'Acetone': 778.0, 'Methanol': 774.0
    },
    333: {
        'Water': 983.2, 'Ethanol': 755.0, 'Propylene_Glycol': 1008.0, 'Mercury': 13438.0, 'Toluene': 832.7,
        'Gasoline': 710.0, 'Kerosene': 790.0, 'Benzene': 839.0, 'Acetone': 765.0, 'Methanol': 756.0
    },
    353: {
        'Water': 971.8, 'Ethanol': 738.0, 'Propylene_Glycol': 992.0, 'Mercury': 13384.0, 'Toluene': 815.6,
        'Gasoline': 700.0, 'Kerosene': 780.0, 'Benzene': 819.0, 'Acetone': 752.0, 'Methanol': 738.0
    }
}
viscosity_ref = {
    273: {
        'Water': 1.79e-3, 'Ethanol': 1.77e-3, 'Propylene_Glycol': 4.20e-2, 'Mercury': 1.66e-3, 'Toluene': 6.90e-4,
        'Gasoline': 6.50e-4, 'Kerosene': 2.40e-3, 'Benzene': 7.56e-4, 'Acetone': 3.94e-4, 'Methanol': 8.17e-4
    },
    293: {
        'Water': 1.00e-3, 'Ethanol': 1.20e-3, 'Propylene_Glycol': 1.60e-2, 'Mercury': 1.55e-3, 'Toluene': 5.59e-4,
        'Gasoline': 5.00e-4, 'Kerosene': 1.60e-3, 'Benzene': 6.03e-4, 'Acetone': 3.16e-4, 'Methanol': 5.83e-4
    },
    313: {
        'Water': 6.53e-4, 'Ethanol': 8.30e-4, 'Propylene_Glycol': 7.00e-3, 'Mercury': 1.45e-3, 'Toluene': 4.58e-4,
        'Gasoline': 4.00e-4, 'Kerosene': 1.10e-3, 'Benzene': 4.87e-4, 'Acetone': 2.58e-4, 'Methanol': 4.32e-4
    },
    333: {
        'Water': 4.67e-4, 'Ethanol': 6.00e-4, 'Propylene_Glycol': 3.50e-3, 'Mercury': 1.36e-3, 'Toluene': 3.80e-4,
        'Gasoline': 3.30e-4, 'Kerosene': 8.00e-4, 'Benzene': 3.99e-4, 'Acetone': 2.14e-4, 'Methanol': 3.31e-4
    },
    353: {
        'Water': 3.55e-4, 'Ethanol': 4.50e-4, 'Propylene_Glycol': 1.90e-3, 'Mercury': 1.28e-3, 'Toluene': 3.20e-4,
        'Gasoline': 2.80e-4, 'Kerosene': 6.00e-4, 'Benzene': 3.33e-4, 'Acetone': 1.80e-4, 'Methanol': 2.61e-4
    }
}
boiling_points = {
    'Water': 373.15, 'Ethanol': 351.5, 'Propylene_Glycol': 461.6, 'Mercury': 629.8, 'Toluene': 383.8,
    'Gasoline': 343.0, 'Kerosene': 423.0, 'Benzene': 353.2, 'Acetone': 329.2, 'Methanol': 337.7
}

reference_temps = [273, 293, 313, 333, 353]

In [3]:
def interpolate_property(T, ref_temps, prop_dict, liquid):
    """Interpolate density or viscosity with quadratic smoothing."""
    T = np.clip(T, min(ref_temps), max(ref_temps))
    for i in range(len(ref_temps) - 1):
        T1, T2 = ref_temps[i], ref_temps[i + 1]
        if T1 <= T <= T2:
            p1, p2 = prop_dict[T1][liquid], prop_dict[T2][liquid]
            frac = (T - T1) / (T2 - T1)
            return p1 + (p2 - p1) * frac * (2 - frac)
    return prop_dict[ref_temps[-1]][liquid]

def vapor_pressure(liquid, T):
    """Estimate vapor pressure using Clausius-Clapeyron approximation."""
    Tb = boiling_points[liquid]
    P_vap = 101325 * np.exp(-5000 * (1/T - 1/Tb))
    return P_vap

def is_liquid(row):
    """Stricter check for liquid state."""
    Tb = boiling_points[row['Liquid']]
    P_vap = vapor_pressure(row['Liquid'], row['Temperature'])
    return (row['Temperature'] < 0.85 * Tb) and (row['Pressure'] > 2.5 * P_vap)

def generate_data(n, min_re, max_re, velocity_range, diameter_range, liquids_subset, temp_range=(273, 353)):
    """Generate liquid flow data without log transformations."""
    batch_size = 100000
    df_list = []
    features = ['Density', 'Velocity', 'Diameter', 'Viscosity', 'Re']

    while len(df_list) * batch_size < n:
        data = {
            'Liquid': np.random.choice(liquids_subset, batch_size),
            'Temperature': np.random.uniform(temp_range[0], temp_range[1], batch_size),
            'Pressure': np.random.uniform(0.8, 7.0, batch_size) * 101325,
            'Velocity': np.random.uniform(velocity_range[0], velocity_range[1], batch_size),
            'Diameter': np.random.uniform(diameter_range[0], diameter_range[1], batch_size)
        }
        df = pd.DataFrame(data)

        # Calculate density and viscosity
        df['Density'] = df.apply(lambda x: interpolate_property(x['Temperature'], reference_temps, density_ref, x['Liquid']), axis=1)
        df['Viscosity'] = df.apply(lambda x: interpolate_property(x['Temperature'], reference_temps, viscosity_ref, x['Liquid']), axis=1)

        # Calculate Re
        df['Re'] = df['Density'] * df['Velocity'] * df['Diameter'] / df['Viscosity']

        # Assign flow regime
        df['Flow_Regime'] = df['Re'].apply(lambda x: 'Laminar' if x < 2300 else 'Transitional' if x < 4000 else 'Turbulent')

        # Filter by Re range and phase
        df = df[(df['Re'] >= min_re) & (df['Re'] < max_re)]
        df = df[df.apply(is_liquid, axis=1)]

        # Outlier filtering (z-score < 3)
        z_scores = np.abs(stats.zscore(df[features]))
        df = df[(z_scores < 3).all(axis=1)]

        if not df.empty:
            df_list.append(df)

    # Combine batches
    df = pd.concat(df_list, ignore_index=True)

    # Trim to desired size
    if len(df) > n:
        df = df.sample(n=n, random_state=42)

    return df

In [6]:
# Parameters
n_per_regime = 5000000
target_samples = 150000
np.random.seed(42)

In [7]:
# Standard liquids (exclude Propylene_Glycol for separate handling)
standard_liquids = [l for l in liquids if l != 'Propylene_Glycol']
# Laminar: Re < 2300
laminar_df = generate_data(n_per_regime, 0, 2300, velocity_range=(0.005, 0.25), diameter_range=(0.0003, 0.007), liquids_subset=standard_liquids)
# Transitional: 2300 <= Re < 4000
transitional_df = generate_data(n_per_regime, 2300, 4000, velocity_range=(0.03, 0.7), diameter_range=(0.0025, 0.035), liquids_subset=standard_liquids)
# Turbulent: Re >= 4000
turbulent_df = generate_data(n_per_regime, 4000, 1e7, velocity_range=(0.25, 7.0), diameter_range=(0.02, 0.35), liquids_subset=standard_liquids)

In [8]:
# High-viscosity liquid (Propylene_Glycol)
high_visc_liquids = ['Propylene_Glycol']
laminar_high_visc = generate_data(n_per_regime, 0, 2300, velocity_range=(0.01, 0.4), diameter_range=(0.001, 0.015), liquids_subset=high_visc_liquids)
transitional_high_visc = generate_data(n_per_regime, 2300, 4000, velocity_range=(0.06, 1.8), diameter_range=(0.006, 0.05), liquids_subset=high_visc_liquids)
turbulent_high_visc = generate_data(n_per_regime, 4000, 1e7, velocity_range=(0.6, 12.0), diameter_range=(0.025, 0.25), liquids_subset=high_visc_liquids)

In [9]:
# Combine datasets
df_list = [laminar_df, transitional_df, turbulent_df, laminar_high_visc, transitional_high_visc, turbulent_high_visc]

In [10]:
# Subsample to balance regimes and liquids
def balance_liquids(df, target_per_liquid=15000):  # 150000 / 10 liquids = 15000
    liquid_dfs = []
    for liquid in liquids:
        liquid_df = df[df['Liquid'] == liquid]
        n_samples = min(len(liquid_df), target_per_liquid)
        if n_samples > 0:
            liquid_dfs.append(liquid_df.sample(n=n_samples, random_state=42))
    return pd.concat(liquid_dfs, ignore_index=True)

In [11]:
# Combine standard and high-viscosity liquids
laminar_df = balance_liquids(pd.concat([df_list[0], df_list[3]], ignore_index=True))
transitional_df = balance_liquids(pd.concat([df_list[1], df_list[4]], ignore_index=True))
turbulent_df = balance_liquids(pd.concat([df_list[2], df_list[5]], ignore_index=True))

In [12]:
# Combine final dataset
df = pd.concat([laminar_df, transitional_df, turbulent_df], ignore_index=True)

In [13]:
# Physical constraints to remove outliers
df = df[df['Re'] < 1e7]
df = df[df['Pressure'] <= 7 * 101325]
df = df[df['Velocity'] < 25]
df = df[df['Density'] > 0]
df = df[df['Viscosity'] > 0]

In [14]:
# Statistical outlier removal
features = ['Density', 'Velocity', 'Diameter', 'Viscosity', 'Re']
z_scores = np.abs(stats.zscore(df[features]))
df = df[(z_scores < 3).all(axis=1)]

In [15]:
# Ensure exactly 150,000 samples per regime
def balance_regimes(df, regime, target):
    regime_df = df[df['Flow_Regime'] == regime]
    if len(regime_df) > target:
        return regime_df.sample(n=target, random_state=42)
    return regime_df

In [16]:
laminar_final = balance_regimes(df, 'Laminar', 150000)
transitional_final = balance_regimes(df, 'Transitional', 150000)
turbulent_final = balance_regimes(df, 'Turbulent', 150000)

In [17]:
# Final dataset
df_final = pd.concat([laminar_final, transitional_final, turbulent_final], ignore_index=True)

In [18]:
df_final

,Liquid,Temperature,Pressure,Velocity,Diameter,Density,Viscosity,Re,Flow_Regime
0,Water,303.887734,566577.089201,0.167911,0.001766,993.445501,0.000725,4.062100e+02,Laminar
1,Water,300.330493,526151.231209,0.240376,0.005375,994.607746,0.000792,1.622136e+03,Laminar
2,Water,289.115836,406932.394822,0.245659,0.003284,998.260347,0.001030,7.819970e+02,Laminar
3,Water,310.447332,128628.489890,0.221774,0.001351,992.297742,0.000659,4.513204e+02,Laminar
4,Water,279.455156,267227.789632,0.228751,0.004606,998.933851,0.001362,7.725014e+02,Laminar
...,...,...,...,...,...,...,...,...,...
374863,Methanol,273.107036,433463.432343,2.521798,0.192172,809.807850,0.000815,4.818252e+05,Turbulent
374864,Methanol,280.856433,227408.772030,4.382561,0.207311,798.635980,0.000669,1.084177e+06,Turbulent
374865,Methanol,278.817619,353823.455534,1.064203,0.134100,801.051297,0.000701,1.631552e+05,Turbulent
374866,Methanol,286.517062,202943.331984,4.163458,0.308471,793.891282,0.000608,1.678112e+06,Turbulent


In [65]:
# Check for invalid values
print("\nNaN Values:")
print(df_final.isna().sum())
print("\nInfinite Values:")
print(np.isinf(df_final[features]).sum())


NaN Values:
Liquid               0
Temperature          0
Pressure             0
Velocity             0
Diameter             0
Density              0
Viscosity            0
Re                   0
Flow_Regime          0
Near_Vaporization    0
dtype: int64

Infinite Values:
Density      0
Velocity     0
Diameter     0
Viscosity    0
Re           0
dtype: int64


In [21]:
print("\nFlow Regime Distribution:")
print(df_final['Flow_Regime'].value_counts())


Flow Regime Distribution:
Flow_Regime
Laminar         130709
Turbulent       123845
Transitional    120314
Name: count, dtype: int64


In [22]:
print("\nLiquid Distribution:")
print(df_final['Liquid'].value_counts())


Liquid Distribution:
Liquid
Ethanol             45000
Kerosene            44992
Gasoline            44872
Water               44864
Benzene             44773
Methanol            44411
Toluene             44127
Acetone             32549
Propylene_Glycol    29280
Name: count, dtype: int64


In [63]:
# Check near vaporization
df_final['Near_Vaporization'] = df_final.apply(lambda row: (row['Temperature'] > 0.9 * boiling_points[row['Liquid']]) or (row['Pressure'] < 2.0 * vapor_pressure(row['Liquid'], row['Temperature'])), axis=1)
print("\nSamples Near Vaporization:")
print(df_final['Near_Vaporization'].value_counts())


Samples Near Vaporization:
Near_Vaporization
False    374868
Name: count, dtype: int64


In [64]:
# Feature correlations
print("\nFeature Correlations:")
corr_matrix = df_final[features].corr()
print(corr_matrix)


Feature Correlations:
            Density  Velocity  Diameter  Viscosity        Re
Density    1.000000  0.005164 -0.037230   0.492151 -0.065789
Velocity   0.005164  1.000000  0.656699   0.015071  0.769325
Diameter  -0.037230  0.656699  1.000000  -0.048954  0.782411
Viscosity  0.492151  0.015071 -0.048954   1.000000 -0.138646
Re        -0.065789  0.769325  0.782411  -0.138646  1.000000


In [23]:
# Save cleaned dataset
df_final.to_csv('reynolds_liquid_dataset.csv', index=False)

In [24]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from scipy import stats

In [25]:
liq_df= pd.read_csv('/content/reynolds_liquid_dataset.csv')

In [26]:
liq_df

,Liquid,Temperature,Pressure,Velocity,Diameter,Density,Viscosity,Re,Flow_Regime
0,Water,303.887734,566577.089201,0.167911,0.001766,993.445501,0.000725,4.062100e+02,Laminar
1,Water,300.330493,526151.231209,0.240376,0.005375,994.607746,0.000792,1.622136e+03,Laminar
2,Water,289.115836,406932.394822,0.245659,0.003284,998.260347,0.001030,7.819970e+02,Laminar
3,Water,310.447332,128628.489890,0.221774,0.001351,992.297742,0.000659,4.513204e+02,Laminar
4,Water,279.455156,267227.789632,0.228751,0.004606,998.933851,0.001362,7.725014e+02,Laminar
...,...,...,...,...,...,...,...,...,...
374863,Methanol,273.107036,433463.432343,2.521798,0.192172,809.807850,0.000815,4.818252e+05,Turbulent
374864,Methanol,280.856433,227408.772030,4.382561,0.207311,798.635980,0.000669,1.084177e+06,Turbulent
374865,Methanol,278.817619,353823.455534,1.064203,0.134100,801.051297,0.000701,1.631552e+05,Turbulent
374866,Methanol,286.517062,202943.331984,4.163458,0.308471,793.891282,0.000608,1.678112e+06,Turbulent


In [27]:
X = liq_df[['Density', 'Velocity', 'Diameter', 'Viscosity']].values
y = liq_df['Re'].values

In [28]:
X_log = np.log1p(X)
y_log = np.log1p(y)

In [29]:
print("NaN in X:", np.any(np.isnan(X)))
print("NaN in y:", np.any(np.isnan(y)))
print("Infinite in X:", np.any(np.isinf(X)))
print("Infinite in y:", np.any(np.isinf(y)))

NaN in X: False
NaN in y: False
Infinite in X: False
Infinite in y: False


In [30]:
print("Min Re:", np.min(y))
print("Max Re:", np.max(y))
print("Min log(Re):", np.min(y_log))
print("Max log(Re):", np.max(y_log))

Min Re: 1.149691527050846
Max Re: 2919507.048039416
Min log(Re): 0.7653243560556315
Max log(Re): 14.88692568335184


In [31]:
z_scores = np.abs(stats.zscore(X_log))
outliers = (z_scores > 3).any(axis=1)
print(f"Number of outliers: {np.sum(outliers)}")

Number of outliers: 11906


In [34]:
# Filter out outliers
liq_df_clean = liq_df[~outliers]
X_clean = X[~outliers]
y_clean = y[~outliers]
X_log_clean = X_log[~outliers]
y_log_clean = y_log[~outliers]

# Verify dataset after outlier removal
print("Dataset Shape after Outlier Removal:", liq_df_clean.shape)
print("NaN in X_clean:", np.any(np.isnan(X_clean)))
print("NaN in y_clean:", np.any(np.isnan(y_clean)))
print("Infinite in X_clean:", np.any(np.isinf(X_clean)))
print("Infinite in y_clean:", np.any(np.isinf(y_clean)))

Dataset Shape after Outlier Removal: (362962, 9)
NaN in X_clean: False
NaN in y_clean: False
Infinite in X_clean: False
Infinite in y_clean: False


In [36]:
feature_scaler = MinMaxScaler()
X_scaled = feature_scaler.fit_transform(X_log_clean)
target_scaler = MinMaxScaler()
y_scaled = target_scaler.fit_transform(y_log_clean.reshape(-1, 1)).flatten()

In [37]:
# Display scaled data statistics
print("\nScaled Features Statistics:")
print(pd.DataFrame(X_scaled, columns=['Density', 'Velocity', 'Diameter', 'Viscosity']).describe())
print("\nScaled Target Statistics:")
print(pd.Series(y_scaled, name='Re').describe())


Scaled Features Statistics:
             Density       Velocity       Diameter      Viscosity
count  362962.000000  362962.000000  362962.000000  362962.000000
mean        0.414931       0.268749       0.200039       0.105350
std         0.285858       0.297806       0.286638       0.129945
min         0.000000       0.000000       0.000000       0.000000
25%         0.250835       0.054180       0.014175       0.038443
50%         0.289511       0.101299       0.040481       0.057155
75%         0.560548       0.484337       0.316661       0.141411
max         1.000000       1.000000       1.000000       1.000000

Scaled Target Statistics:
count    362962.000000
mean          0.579241
std           0.226416
min           0.000000
25%           0.413424
50%           0.514145
75%           0.822435
max           1.000000
Name: Re, dtype: float64


In [38]:
joblib.dump(feature_scaler, 'scaler_X.pkl')

['scaler_X.pkl']

In [39]:
joblib.dump(target_scaler, 'scaler_y.pkl')

['scaler_y.pkl']

In [41]:
liq_df_clean.to_csv('reynolds_liq_dataset_clean.csv', index=False)

In [42]:
print("X_scaled min:", np.min(X_scaled, axis=0))
print("X_scaled max:", np.max(X_scaled, axis=0))
print("y_scaled min:", np.min(y_scaled))
print("y_scaled max:", np.max(y_scaled))

X_scaled min: [0. 0. 0. 0.]
X_scaled max: [1. 1. 1. 1.]
y_scaled min: 0.0
y_scaled max: 0.9999999999999999


In [43]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

In [44]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((290369, 4), (72593, 4), (290369,), (72593,))

In [45]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(1)
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [54]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

In [55]:
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=5, # Stop after 5 epochs with no improvement
    min_delta=0.001,
    restore_best_weights=True  # Restore weights from the best epoch
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1,
    callbacks=[early_stopping]  # Add early stopping callback
)

In [ ]:
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()

In [58]:
loss, mae = model.evaluate(X_test, y_test)
print(f"Test Loss (MSE): {loss:.6f}, Test MAE: {mae:.6f}")

2269/2269 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 9.4915e-05 - mae: 0.0077
Test Loss (MSE): 0.000096, Test MAE: 0.007758


In [59]:
# Evaluate the model on the test set
y_pred_scaled = model.predict(X_test)

# Inverse transform predictions and true values to log scale
y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_test_orig = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

2269/2269 ━━━━━━━━━━━━━━━━━━━━ 2s 992us/step


In [60]:
# Calculate metrics on log scale
rmse_log = np.sqrt(mean_squared_error(y_test_orig, y_pred))
r2_log = r2_score(y_test_orig, y_pred)
print(f"\nTest RMSE (log scale): {rmse_log}")
print(f"Test R² (log scale): {r2_log}")

# Convert predictions and true values back to original Re scale
y_pred_orig = np.expm1(y_pred)  # Inverse of log1p
y_test_orig_re = np.expm1(y_test_orig)  # Inverse of log1p

# Calculate metrics on original Re scale
rmse_orig = np.sqrt(mean_squared_error(y_test_orig_re, y_pred_orig))
r2_orig = r2_score(y_test_orig_re, y_pred_orig)
print(f"Test RMSE (original Re scale): {rmse_orig}")
print(f"Test R² (original Re scale): {r2_orig}")


Test RMSE (log scale): 0.13815551138442084
Test R² (log scale): 0.9981364465807219
Test RMSE (original Re scale): 60131.569024046796
Test R² (original Re scale): 0.9870172121343991


In [61]:
y_pred_scaled = model.predict(X_test, verbose=0)
y_pred_log = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
y_pred = np.expm1(y_pred_log)
y_test_log = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_test_original = np.expm1(y_test_log)

ss_total = np.sum((y_test_original - np.mean(y_test_original)) ** 2)
ss_residual = np.sum((y_test_original - y_pred) ** 2)
r2 = 1 - (ss_residual / ss_total)
print(f"R² Score: {r2:.6f}")

epsilon = 1e-10
mape = 100 * np.mean(np.abs((y_test_original - y_pred) / (y_test_original + epsilon)))
print(f"MAPE (%): {mape:.2f}%")

mse_original = np.mean((y_pred - y_test_original) ** 2)
mae_original = np.mean(np.abs(y_pred - y_test_original))
print(f"MSE on original scale: {mse_original:.2f}")
print(f"MAE on original scale: {mae_original:.2f}")

print("\nSample Predictions vs Actual Values:")
for i in range(5):
    print(f"Predicted Re: {y_pred[i]:.2f}, Actual Re: {y_test_original[i]:.2f}")

R² Score: 0.987017
MAPE (%): 10.31%
MSE on original scale: 3615805593.29
MAE on original scale: 23499.14

Sample Predictions vs Actual Values:
Predicted Re: 2953.72, Actual Re: 3245.65
Predicted Re: 748.09, Actual Re: 860.11
Predicted Re: 2343.12, Actual Re: 2644.18
Predicted Re: 47.92, Actual Re: 47.90
Predicted Re: 3727.83, Actual Re: 3509.98


In [62]:
# Save model and scalers
model.save('reynolds_model_liq.h5')
joblib.dump(feature_scaler, 'feature_scaler_liq.pkl')
joblib.dump(target_scaler, 'target_scaler_liq.pkl')

['target_scaler_liq.pkl']

In [67]:
import json

In [66]:
liquid_df = pd.read_csv('reynolds_liq_dataset_clean.csv')

In [68]:
# Features for sliders
features = ['Density', 'Velocity', 'Diameter', 'Viscosity']

# Compute min and max with a 5% buffer
ranges = {'liquid': {}}

for feature in features:
    min_val = liquid_df[feature].min()
    max_val = liquid_df[feature].max()
    buffer = 0.05 * (max_val - min_val)
    ranges['liquid'][feature] = {
        'min': max(0, min_val - buffer),  # Ensure non-negative
        'max': max_val + buffer,
        'step': (max_val - min_val) / 1000  # Fine-grained steps
    }

In [69]:
# Save ranges to JSON
with open('liquid_slider_ranges.json', 'w') as f:
    json.dump(ranges, f, indent=4)

In [70]:
# Display summary statistics for verification
print("\nSummary Statistics of Features:")
print(liquid_df[features].describe())


Summary Statistics of Features:
             Density       Velocity       Diameter      Viscosity
count  362962.000000  362962.000000  362962.000000  362962.000000
mean      842.128801       1.339336       0.066216       0.000924
std        82.783321       1.949563       0.097308       0.000708
min       730.052589       0.005000       0.000300       0.000350
25%       793.665025       0.132086       0.004560       0.000559
50%       803.954252       0.255605       0.012514       0.000661
75%       879.896592       1.913748       0.099918       0.001120
max      1018.548568       8.049758       0.349997       0.005807


In [72]:
# Print ranges for verification
print("Slider Ranges for Liquid Dataset:")
print(json.dumps(ranges, indent=4))

Slider Ranges for Liquid Dataset:
{
    "liquid": {
        "Density": {
            "min": 715.6277903038863,
            "max": 1032.9733671921813,
            "step": 0.2884959789893592
        },
        "Velocity": {
            "min": 0,
            "max": 8.451995705516785,
            "step": 0.008044757410692683
        },
        "Diameter": {
            "min": 0,
            "max": 0.36748163999068206,
            "step": 0.00034969670529363977
        },
        "Viscosity": {
            "min": 7.699317799509498e-05,
            "max": 0.006080380547145805,
            "step": 5.4576248810461e-06
        }
    }
}
